In [11]:
import os
import dotenv
import pandas as pd
from entsoe import EntsoePandasClient
from entsoe.exceptions import NoMatchingDataError

dotenv.load_dotenv()

api_key = os.getenv("ENTSO_API_KEY")

In [ ]:
client = EntsoePandasClient(api_key=api_key)

start = pd.Timestamp("2024-10-01", tz="Europe/Brussels")
end = pd.Timestamp("2026-10-01", tz="Europe/Brussels")


#codes of zones which prices are affecting NO2
price_codes = ["NL", "DE_LU", "BE", "FR"] #possibly BE belgium or FR france or GB great britain?
neighbor_codes = ["NO_5", "DK_1", "NO_1", "NL", "DE_LU"]


In [13]:
country_prices = []
for code in price_codes:
    print(code)
    prices = client.query_day_ahead_prices(
        country_code=code,
        start=start,
        end=end
    )
    # query_day_ahead_prices returns a Series named "Price";
    # rename it to "price_<zone>" so concat produces unique columns
    country_prices.append(prices.rename(f"price_{code}"))
country_prices_df = pd.concat(country_prices, axis="columns", sort=False)

NL
DE_LU
BE
FR


In [14]:
no2_code = "NO_2"

# A69
wind_solar_forecast = client.query_wind_and_solar_forecast(
    country_code=no2_code,
    start=start,
    end=end
)

# A72 -> returns a pandas Series
no2_reservoirs = client.query_aggregate_water_reservoirs_and_hydro_storage(
    country_code=no2_code,
    start=start,
    end=end
)
# A72 -> returns a pandas Series
no1_reservoirs = client.query_aggregate_water_reservoirs_and_hydro_storage(
    country_code="NO_1",
    start=start,
    end=end
)
# A72 -> returns a pandas Series
no5_reservoirs = client.query_aggregate_water_reservoirs_and_hydro_storage(
    country_code="NO_5",
    start=start,
    end=end
)

# A75
generation_forecast = client.query_generation_forecast(
    country_code=no2_code,
    start=start,
    end=end
)

# entsoe-py returns a Series whose name ends with "- Actual Aggregated";
# relabel it as a forecast since document A75 contains day-ahead
generation_forecast.name = "Forecasted Generation"

# A65
load_forecast = client.query_load_forecast(
    country_code=no2_code,
    start=start,
    end=end
)

combined_df = pd.concat(
    [country_prices_df,wind_solar_forecast, generation_forecast, load_forecast], axis="columns", sort=False
)

combined_df["no2_reservoir_level_MWh"] = no2_reservoirs.reindex(
    combined_df.index,
    method="ffill"
)

combined_df["no1_reservoir_level_MWh"] = no1_reservoirs.reindex(
    combined_df.index,
    method="ffill"
)

combined_df["no5_reservoir_level_MWh"] = no5_reservoirs.reindex(
    combined_df.index,
    method="ffill"
)


In [ ]:
def query_entsoe_flows_and_capacities(code_from, code_to, start, end):
    scheduled = None
    net_transfer_capacity = None
    offered = None
    print(f"getting flows and capacities from {code_from} to {code_to}:")
    try:
        scheduled = client.query_scheduled_exchanges(
            country_code_from=code_from,
            country_code_to=code_to,
            start=start,
            end=end,
            day_ahead=True
        )
        scheduled.name = f"Scheduled Exchanges from {code_from} to {code_to}"
    except NoMatchingDataError:
        print(f"No matching data for scheduled exchanges from {code_from} to {code_to}")
    try:
        net_transfer_capacity = client.query_net_transfer_capacity_dayahead(
            country_code_from=code_from,
            country_code_to=code_to,
            start=start,
            end=end
        )
        net_transfer_capacity.name = f"Net Transfer Capacity from {code_from} to {code_to}"
    except NoMatchingDataError:
        print(f"No matching data for net transfer capacity from {code_from} to {code_to}")
    try:
        offered = client.query_offered_capacity(
            country_code_from=code_from,
            country_code_to=code_to,
            contract_marketagreement_type='A01', #A01 is the type for day-ahead market
            start=start,
            end=end,
            implicit=True
        )
        offered.name = f"Offered Capacity from {code_from} to {code_to}"
    except NoMatchingDataError:
        print(f"No matching data for offered capacity from {code_from} to {code_to}")
    return scheduled, net_transfer_capacity, offered


In [ ]:
all_flows_and_capacities = pd.DataFrame()
for code in neighbor_codes:
    flows_and_capacities_from_no2 = query_entsoe_flows_and_capacities(
        code_from="NO_2",
        code_to=code,
        start=start,
        end=end
    )
    flows_and_capacities_to_no2 = query_entsoe_flows_and_capacities(
        code_from=code,
        code_to="NO_2",
        start=start,
        end=end
    )

    flows_and_capacities_from_no2 = pd.concat(flows_and_capacities_from_no2, axis="columns", sort=False)
    flows_and_capacities_to_no2 = pd.concat(flows_and_capacities_to_no2, axis="columns", sort=False)
    flows_and_capacities = pd.concat([flows_and_capacities_from_no2, flows_and_capacities_to_no2], axis="columns", sort=False)
    print(f"Flows and capacities between NO_2 and {code}:", flows_and_capacities.head())
    all_flows_and_capacities = pd.concat([all_flows_and_capacities, flows_and_capacities], axis="columns", sort=False)

In [15]:
print(combined_df.head())
#print(all_flows_and_capacities.head())

#combined_df = pd.concat([combined_df, all_flows_and_capacities], axis="columns", sort=False)
#fill missing values with forward fill
combined_df = combined_df.ffill()
#take average of the 4 quarters in the hourly data to get hourly data
combined_df = combined_df.resample('h').mean()

combined_df.to_csv("entsoe_data_hourly.csv")

                           price_NL  price_DE_LU  price_BE  price_FR  Solar  \
2024-09-30 22:00:00+00:00      3.21         3.21      3.21      3.21    0.0   
2024-09-30 23:00:00+00:00      0.07         0.07      0.07      0.07    0.0   
2024-10-01 00:00:00+00:00      0.05         0.05      0.05      0.05    0.0   
2024-10-01 01:00:00+00:00      0.02         0.02      0.02      0.02    0.0   
2024-10-01 02:00:00+00:00      0.09         0.09      0.09      0.09    0.0   

                           Wind Onshore  Wind Offshore  Forecasted Generation  \
2024-09-30 22:00:00+00:00        949.94            NaN                2245.80   
2024-09-30 23:00:00+00:00        926.94            NaN                2024.80   
2024-10-01 00:00:00+00:00        897.66            NaN                2169.23   
2024-10-01 01:00:00+00:00        866.61            NaN                2134.06   
2024-10-01 02:00:00+00:00        837.23            NaN                2325.65   

                           Forecasted 